# LLM-WITHMEM: Level 4 Multi-Bank Memory Encoder Training

Train a **51.3M-parameter** multi-bank query-conditioned memory encoder on **A100 40GB**.

**Architecture**:
- Profile → SharedEncoder (4-layer) → 5 PerceiverBankHeads (episodic/semantic/procedural/emotional/prospective)
- Query → QueryEncoder (2-layer) → attention pool → query_vector
- WorkingMemory: query-conditioned cross-attention over all 72 bank slots → 32 output vectors
- DynamicGateNetwork: query → per-layer per-head sigmoid gates via MLP
- 4 layer-group K,V projections × gates → inject into SmolLM2-1.7B DynamicCache

**Training**: KL Distillation — gold prompt has ONLY query-relevant memory types, so the encoder learns which banks to activate per query.

**A100 40GB Config**: batch_size=4, grad_accum=2, effective batch=8, bf16/fp16 mixed

## 1. GPU Check & A100 Auto-Config

In [ ]:
import torch
import subprocess

assert torch.cuda.is_available(), "No GPU detected — switch to a GPU runtime!"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# ═══ Auto-select batch size based on GPU VRAM ═══
if vram_gb >= 70:     # H100 / A100 80GB
    BATCH_SIZE = 8
    GRAD_ACCUM = 1
elif vram_gb >= 38:   # A100 40GB ← your target
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
elif vram_gb >= 20:   # A10G / RTX 4090
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
elif vram_gb >= 14:   # T4 / V100
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8

print(f"\n✓ Auto-config: batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

# Enable TF32 for A100 matmul speedup
if "A100" in gpu_name or "H100" in gpu_name:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("  ✓ TF32 enabled for A100/H100")